## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
import numpy as np
import matplotlib.pylab as plt
from sklearn.decomposition import PCA
import math
import requests
import pathlib
from functools import reduce

plt.rcParams["font.family"] = "Arial"
sns.set_theme(style="white")

## PCA

In [ ]:
# specify file path
data_path = pathlib.Path('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/02_RNA-seq')
df_path = data_path / "normalized_counts.tsv"

In [ ]:
def get_condition_macrophage(filename):
    condition = filename.split('_')[-1]
    condition_dictionary = {
        '1': 'M0',
        '2': 'TLR1-2',
        '3': 'TLR3',
        '4': 'TLR4',
        '5': 'TLR7',
        '6': 'TLR8',
        '7': 'TLR9',
        '8': 'STING',
        '9': 'M0 High'
    }
    return condition_dictionary[condition]


# map ensembl genes to gene symbols

def map_ensembl_to_symbol(df, gene_col_name = "genes"):
    print(gene_col_name)
    r = requests.post(
        url='https://biit.cs.ut.ee/gprofiler/api/convert/convert/',
        json={
            'organism':'hsapiens',
            'target':'ENSG',
            'query':list(df[gene_col_name]),
        }
    )
    gene_to_symbol = dict()
    for entry in r.json()["result"]:
        if entry["name"] == "None":
            # unnamed genes can use ensembl id
            gene_to_symbol[entry["incoming"]] = entry["incoming"]
        elif entry["incoming"] not in gene_to_symbol:
            gene_to_symbol[entry["incoming"]] = entry["name"]
    df["Name"] = df[gene_col_name].apply(lambda x: gene_to_symbol[x])
    return df

In [ ]:
import sys
sys.path.append("..")
from src.pca_utils import run_pca, percent_explained_df


def run_and_plot_pca(data_df, output_path):
    pathlib.Path(output_path).mkdir(exist_ok = True)
    # run centralized PCA core (log2 -> drop non-finite rows -> transpose -> PCA(5))
    principal_components_df, loadings_df, explained_variance_ratio = run_pca(data_df, n_components=5)
    number_of_proteins_in_common = loadings_df.shape[0]
    # add metadata information
    principal_components_df["condition"] = (
        principal_components_df["sample_name"]
        .apply(
            get_condition_macrophage))
    principal_components_df
    # create barplot of explained variance
    sns.barplot(x=['PC1','PC2','PC3','PC4', 'PC5'],
            y=(explained_variance_ratio)*100,
            edgecolor = 'black',
            palette = 'YlGnBu').set(title='Principal Components', ylabel='Percent (%)')
    plt.show()
    plt.savefig(output_path / "principal_components.svg")
    # specify range of plot, symbold and colors
    
    symbols = ["circle", "triangle-up"]
    principal_components_df.to_csv(output_path / "principal_components.csv")
    # self-updating PC% axis labels for wp_visualization.Rmd
    percent_explained_df(explained_variance_ratio).to_csv(output_path / "percent_explained.csv", index=False)
    # create pca plot
    fig = px.scatter(principal_components_df,
                    x='PC1',
                    y='PC2',
                    hover_data=['sample_name'],
                    color=principal_components_df['condition'],
                    labels={"PC1": "PC1 ({}%)".format(round((explained_variance_ratio[0] * 100),2)),
                            "PC2": "PC2 ({}%)".format(round((explained_variance_ratio[1] * 100),2)),
                            "species": "Species"},
                    title="PCA plot" +" ("+str(number_of_proteins_in_common)+" Genes)",
                    template="plotly_white",
                    category_orders={"condition": ['D2', 'D4A', 'D4C', 'D8A', 'D8C']})
    
    #updates the range of x and y axis
    fig.update_xaxes(dtick=4, range=[principal_components_df["PC1"].min() - 2, principal_components_df["PC1"].max() + 2])
    fig.update_yaxes(dtick=4, range=[principal_components_df["PC2"].min() - 2, principal_components_df["PC2"].max() + 2])
    
    #determines if border of plot should be shown
    fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    
    #determines gridlines details
    fig.update_xaxes(showgrid=True, gridwidth=2, gridcolor='#E8E8E8')
    fig.update_yaxes(showgrid=True, gridwidth=2, gridcolor='#E8E8E8')
    
    #determines how zerolines should be displayed
    fig.update_xaxes(zerolinewidth=2, zerolinecolor='#E8E8E8')
    fig.update_yaxes(zerolinewidth=2, zerolinecolor='#E8E8E8')
    
    #updates size of feature points in plot
    fig.update_traces(marker=dict(size=12),
                    selector=dict(mode='markers'))
    
    #specifies layout details
    fig.update_layout(height=500,
                    width=600,
                    showlegend=True,
                    legend_title_text='Condition',
                    font=dict(family="Arial",
                              color="black"),
                    title_x=0.45)
    
    fig.write_html(output_path / "pca_plot.html")
    fig.write_image(output_path / "pca_plot.svg")
    #fig.savefig("pca_plot.png")
    fig.show()
    # get one laodings worth, rememeber sum of squares per PC loading = 1
    one_loading_value = math.sqrt(1/len(loadings_df))
    # then filter out ones that contribute more than one variables worth
    filtered_loadings = loadings_df[(loadings_df["PC1"] > one_loading_value) | (loadings_df["PC1"] < -one_loading_value) | (loadings_df["PC2"] > one_loading_value) | (loadings_df["PC2"] < -one_loading_value)]
    
    top_pos_PC1_df = filtered_loadings.sort_values('PC1', ascending=False).head(10)
    top_neg_PC1_df = filtered_loadings.sort_values('PC1').head(10)
    
    top_pos_PC2_df = filtered_loadings.sort_values('PC2', ascending=False).head(10)
    top_neg_PC2_df = filtered_loadings.sort_values('PC2').head(10)
    
    df_list = [top_pos_PC1_df, top_neg_PC1_df, top_pos_PC2_df, top_neg_PC2_df]
    
    merged_loadings_df = reduce(lambda x, y: pd.merge(x[["variable", "PC1", "PC2"]], y[["variable", "PC1", "PC2"]], on = ["variable", "PC1", "PC2"],  how = "outer"), df_list)
    merged_loadings_df.head(5)
    # add loadings to pca plot
    n = merged_loadings_df.shape[0]
    for i in range(n):
        fig.add_annotation(x= 0,
                        y= 0,
                        ax=merged_loadings_df.iloc[i,1] * 80,
                        ay=merged_loadings_df.iloc[i,2] * 80,
                        xref='x',
                        yref='y',
                        axref='x',
                        ayref='y',
                        text=merged_loadings_df.iloc[i,0],
                        showarrow=True,
                        arrowhead=3,
                        arrowsize=1,
                        arrowwidth=1,
                        arrowcolor='red',
                        opacity=0.6,
                        arrowside='start')
    
    fig.write_html(output_path / "pca_plot_with_loadings.html")
    fig.show()
    # create heatmap of loadings P
    plt.figure(figsize=(5,8))
    sns.heatmap(merged_loadings_df.set_index("variable"),
            cmap='YlGnBu',
            linewidths=0.7,
            linecolor="black").set(title='Loadings', ylabel=None)
    plt.savefig(output_path /  "loadings_heatmap.svg", bbox_inches='tight')
    plt.show()
    return (principal_components_df, loadings_df)


In [ ]:
# filter to protein coding genes, then filter to top 5000 highest variance genes
results_dir = data_path / "PCA_5000_genes"
results_dir.mkdir(exist_ok=True)
df = pd.read_csv(df_path, index_col=[0], sep="\t")
df = df.drop(df.filter(like="TZ3").columns, axis = 1)
df = df.drop(df.filter(like="_9").columns, axis = 1)
df = df.drop(df.filter(like="_7").columns, axis = 1)  # drop TLR9 (condition suffix _7) before PCA
gene_id_df = pd.read_csv("mart_export.txt", sep="\t")
protein_coding_genes = gene_id_df[gene_id_df["Gene type"] == "protein_coding"]
gene_stable_ids = set(protein_coding_genes["Gene stable ID version"])
gene_names = set(protein_coding_genes["Gene name"])
gene_synonyms = set(protein_coding_genes["Gene Synonym"])
protein_coding_counts = []
print(df)
for i,r in df.iterrows():
    if (i in gene_stable_ids) or (i in gene_names) or (i in gene_synonyms):
        protein_coding_counts.append(r)
df = pd.DataFrame(protein_coding_counts)
print(df)
df["variance"] = np.var(df, axis = 1)
df = df.sort_values("variance")
df = df.tail(5000)
df = df.drop(["variance"], axis=1)
df.to_csv(results_dir / "normalized_counts_5000_protein_coding_genes.csv")


pc_df, loadings_df = run_and_plot_pca(df, results_dir)

In [ ]:
loadings_df["trimmed_gene_id"] = [x.split(".")[0] for x in loadings_df["variable"]]
loadings_df = map_ensembl_to_symbol(loadings_df, gene_col_name="trimmed_gene_id").drop(["variable", "trimmed_gene_id"], axis =1 )
loadings_df["variable"] = loadings_df["Name"]
loadings_df.drop("Name", axis = 1).to_csv(results_dir / "loadings_df.csv")

## GSEA

In [ ]:
df = pd.read_csv(
    data_path / "unnormalized_counts_filtered.txt",
    sep = "\t"
)

df["trimmed_gene_id"] = [x.split(".")[0] for x in df["gene.id"]]

(
    map_ensembl_to_symbol(df, "trimmed_gene_id")
    .drop(["gene.id","trimmed_gene_id"], axis = 1)
    .set_index("Name")
    .to_csv(data_path / "unnormalized_counts_filtered_mapped.txt", sep = "\t"))